# Chapter 4 (part 5) — Polynomial, Spline, and Generalized Additive Models

Covers: Polynomial and Spline Regression → Generalized Additive Models (GAM), on Ames Housing.

The thread: **"linear regression" means linear in the *coefficients*, not in the raw variables.** Feed it `x²`, `log(x)`, or bendy basis functions and it's still ordinary least squares. The only real question is how much flexibility to allow: too little misses real curvature (bias), too much chases noise (variance). Polynomials bend *globally*, so the tails go wild. Splines bend *locally*, in pieces joined smoothly at knots. A GAM gives each predictor its own spline and adds them up.

Tooling: polynomials via `I(x**2)` in `smf.ols`; splines via patsy `bs()` / `cr()` in formulas; GAMs via `statsmodels.gam.api.GLMGam` + `BSplines` (no `pygam` here).

**How to work each section** (~10 min each; ~1 hour total with the exam): write your one-line prediction in the *Prediction* cell **before** running anything, do the code, then write 2-3 bullets on where the result matched or broke your prediction and why. Ask for a hint if you get stuck.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from patsy import dmatrix
from statsmodels.gam.api import GLMGam, BSplines
from sklearn.model_selection import train_test_split, KFold

pd.set_option("display.max_columns", None)

from mlstats.data import load_ames
ames = load_ames()
ames.shape

## 1. Polynomial Regression

A quadratic adds an `x²` column: `y ≈ b0 + b1*x + b2*x²`, so the slope is no longer constant. Numeric picture: with `b1 = 100` and `b2 = 0.02`, one more unit of x is worth about `100 + 2*0.02*x`: $140 at x = 1,000 and $180 at x = 2,000. `b1` alone is just the slope at x = 0, which may be far outside your data.

Another trap: `Year_Built` is around 1,950, so `Year_Built²` is around 3.8 million and the two columns are almost perfectly correlated. Centering first (`x - x.mean()`) removes most of that without changing the fitted curve.

**Predict first:** will centering change the fitted values? Will an extra year of newness be worth more in 1920 or in 2000?

**Question 1.** (a) Fit `Sale_Price ~ Year_Built + I(Year_Built**2)` raw and with a centered `Year_Built`. Compare the coefficient tables and confirm the fitted values are identical. (b) From the centered model, compute the marginal effect of one more year at 1920, 1960, and 2000. Where does the curve turn?

*Tools for this question:*
- `I(x**2)` inside the formula protects the arithmetic; centered version: create `df["yb_c"] = df["Year_Built"] - df["Year_Built"].mean()` first.
- Marginal effect of a quadratic `b1*x + b2*x**2` is `b1 + 2*b2*x`; get the values with `res.params[...]`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 2. How Flexible Should the Curve Be?

More terms always fit the training data better; the question is held-out performance. Numeric picture: with 5 points, a degree-4 polynomial passes through all of them exactly (zero training error) and swings wildly between them, while a line misses all five a little but stays stable.

**Predict first:** at which degree will CV RMSE bottom out: 1-2, 3-5, or 8+?

**Question 2.** (a) Write a small function `cv_rmse(formula, df)` that returns train RMSE and 5-fold CV RMSE (you'll reuse it in Section 3). Use it on polynomials of standardized `Gr_Liv_Area` for degrees 1, 2, 3, 5, 8, 10 and plot both curves. Which degree would you pick? (b) Plot the degree 2, 5, and 10 fits over `Gr_Liv_Area` from 300 to 7,000 (the data max is 5,642). What happens at the edges and beyond?

*Tools for this question:*
- `KFold(n_splits=5, shuffle=True, random_state=0).split(df)` yields `(train_idx, test_idx)` arrays; use `df.iloc[idx]`.
- Degree-k formula: `"Sale_Price ~ " + " + ".join(["z"] + [f"I(z**{j})" for j in range(2, k + 1)])` where `z` is the standardized column `(x - x.mean()) / x.std()`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 3. Splines

A **spline** cuts the x-axis into pieces at **knots** and fits a separate cubic in each piece, constrained to join smoothly. Numeric picture: knots at 1,000 and 2,000 sqft give three cubics that share the same value, slope, and curvature where they meet. A change in shape above 2,000 doesn't disturb the fit below 1,000, which is the key difference from a polynomial. The **degrees of freedom** (`df`, the number of basis columns) is the flexibility knob. A **natural** spline (`cr`) also forces the curve to be straight outside the outer knots, which tames the tails.

**Predict first:** past 4,000 sqft, which will behave more sensibly, `bs(df=6)` or `cr(df=6)`?

**Question 3.** (a) With `Gr_Liv_Area` as the only predictor, fit `bs(..., df=k)` for k = 4, 6, 12 and `cr(..., df=6)`, and plot them together. Compare `bs` and `cr` past 4,000 sqft. (b) Reuse your `cv_rmse` function (just swap the formula) to compare the best polynomial from Section 2 with `bs` at df 4, 6, 8, 12.

*Tools for this question:*
- `bs(x, df=6, degree=3, lower_bound=lo, upper_bound=hi)` B-spline basis; `cr(x, df=6)` natural cubic spline.
- **Gotcha:** `bs()` can't predict outside the range it was fit on (`Gr_Liv_Area` runs 334 to 5,642). Plot over `np.linspace(col.min(), col.max(), 200)` instead of the 300-7,000 grid from Section 2, or pass `lower_bound` / `upper_bound`; inside CV, compute the bounds from the *full* column.
- Use `cr(x, df=6, constraints="center")`: without it the `cr` columns duplicate the intercept and statsmodels warns that the design matrix is rank-deficient.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 4. Generalized Additive Models

A GAM generalizes the idea: `y ≈ b0 + f1(x1) + f2(x2) + b3*x3`, where each `f` is a spline with a **smoothness penalty** `alpha` (bigger alpha = flatter, more line-like curve). The effects still *add*, so you can plot each `f` on its own and read it like a coefficient: "the contribution of living area to price, holding the other terms fixed". The penalty replaces hand-tuning the number of knots: use plenty of knots and let the penalty decide how much wiggle to keep. Numeric picture: with alpha huge, an 8-df spline is forced nearly straight and behaves like a 1-df line; with alpha near 0 it uses all 8 df.

**Predict first:** on a test set, will the GAM's RMSE gain over the straight-line model be small (under ~15%) or large (over ~15%)? Why?

**Question 4.** Make one 70/30 train/test split. On train, fit `GLMGam` with `BSplines` on `Year_Built` and `Gr_Liv_Area` (`df=[8, 8]`, `degree=[3, 3]`), linear terms for `Garage_Area` and `Total_Bsmt_SF`, and `alpha=[1, 1]`. (a) Plot the partial effect of each smooth. Which is closest to straight? (b) Refit with `alpha` at 1e6 and at 1e-3 and re-plot. What happens, and which knob from Sections 2-3 does alpha replace? (c) Compare test RMSE of the `alpha=[1, 1]` GAM to the straight-line OLS on the same four predictors.

*Tools for this question:*
- `bs = BSplines(df[["Year_Built", "Gr_Liv_Area"]], df=[8, 8], degree=[3, 3])` builds the smoothers.
- `GLMGam.from_formula("Sale_Price ~ Garage_Area + Total_Bsmt_SF", data=df, smoother=bs, alpha=[1, 1])` then `.fit()`; `res.plot_partial(i, cpr=True)` plots smooth number `i`.
- Predict on test: `res.predict(test, exog_smooth=xs)`. **Gotcha:** it raises `NotImplementedError` if any test value falls outside the training range, so clip first: `xs = test[cols].clip(lower=train[cols].min(), upper=train[cols].max(), axis=1)`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

---
# Exam — check your intuition

5 questions, about 3-4 minutes each. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, 2-3 bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Polynomial vs. spline: reading the scoreboard
*Role lean: DS · ML*

One predictor, `Lot_Area`, with a very skewed distribution (a handful of enormous lots).

```text
Fit on 2051 training homes; scored on 879 held-out homes. Predictor is Lot_Area only (range 1,300 to 215,245 sq ft; median 9,436; 35 homes above 30,000).

                           model  train RMSE  test RMSE rank warning
       straight line in Lot_Area       77689      75440             
            polynomial, degree 2       75797      72583             
            polynomial, degree 3       74813      71512             
            polynomial, degree 5       81969      78125          yes
            polynomial, degree 8       83903      79408          yes
    log(Lot_Area), straight line       75116      72671             
     B-spline, 5 df, on Lot_Area       74736      71678             
B-spline, 5 df, on log(Lot_Area)       72340      69675
```

(a) Adding a term to a least-squares fit can never make the *training* error go up — yet look at the training RMSE at degrees 5 and 8. What's going on, and what does the "rank warning" column tell you? (Think about how big and how correlated the columns `Lot_Area`, `Lot_Area^2`, ..., `Lot_Area^8` are, and what a computer does with numbers like 215,245^8.) What would you do to fix it? (b) Between degree 2 and degree 3 the held-out error improves only a little. Using the table, what would you pick and why would "simplest model that's about as good as the best" be a reasonable rule? (c) The 5-df spline on `log(Lot_Area)` beats the 5-df spline on raw `Lot_Area`. Why would transforming first help a spline that can already bend? (d) The straight line in `log(Lot_Area)` has a single slope yet nearly matches the polynomial. What does that say about the *shape* of the true relationship?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — What happens off the edge of the data
*Role lean: ML · DS · interview*

```text
Predicted Sale_Price at a few lot sizes, models fit on all 2,930 homes (largest lot in the data: 215,245):

 Lot_Area (sq ft)  cubic polynomial  straight line in log(Lot_Area)
             1300            113178                           71125
            10000            184003                          187536
            50000            322523                          279368
           100000            236644                          318918
           215245            466864                          362660
           300000           2669997                          381603

For reference, the most expensive home in the data sold for 755,000.
```

(a) Which model's behavior at 300,000 sq ft is plainly broken, and what property of polynomials causes it? (b) The log model looks sensible there. Is it actually *right* at 300,000 sq ft, or just not obviously wrong? What would you need to know to trust any model out there? (c) A natural cubic spline (patsy's `cr()`) is constrained to be *linear* beyond the boundary knots, whereas an ordinary B-spline (`bs()`) in patsy refuses to predict outside the range it was fit on. How does each choice trade safety against honesty about extrapolation? (d) If this were a production pricing model, what guardrail would you add around features like lot size?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Bias, variance, and the number of knots
*Role lean: ML · DS · interview*

```text
Spline of Year_Built (the only predictor) with k degrees of freedom. Each fit uses only 150 random homes; scored on the other 2,780; averaged over 25 random draws.

 spline df  train RMSE  held-out RMSE
         3       61895          63728
         5       61479          64622
         8       61008          65653
        12       60608          70995
        16       60458          84182
```

(a) Describe both columns in words: where is the sweet spot, and why does train RMSE keep dropping while held-out RMSE rises steadily? Which of bias or variance dominates at df = 3, and which at df = 16? (b) With all 2,930 homes in the training set instead of 150, would you expect the best df to move up, down, or stay? Why? (c) Interview version: "How do you choose the number of knots (or the polynomial degree, or the smoothing penalty)? And how is that the same question as choosing the depth of a decision tree?" 

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — A GAM is additive: what does that buy and cost?
*Role lean: DS · ML*

```text
Predicted price for an otherwise-median home (median Gr_Liv_Area = 1442; median Year_Remod_Add = 1993 in the second table; median Total_Bsmt_SF = 990 in the first):

 Year_Remod_Add  homes with that value  additive spline  linear model
           1950                    361           134532        135517
           1960                     29           144056        146390
           1970                     44           146132        157262
           1980                     30           155677        168135
           1990                     29           166405        179007
           2000                    104           175014        189880
           2005                    141           188689        195316
           2010                     13           231824        200752

 Total_Bsmt_SF  homes with at least this much  additive spline  linear model
             0                           2930           136204        121151
           500                           2753           147536        152019
          1000                           1436           169287        182887
          1500                            431           211775        213754
          2000                             58           262667        244622
          3000                              6           343689        306358
          4000                              2           333318        368093
```

(a) The `Year_Remod_Add` column has 361 homes with the value 1950 (the column is floored at 1950: 339 of those 361 are homes built *before* 1950, whose remodel year is recorded as 1950 regardless of what happened). Describe the spline's curve for `Year_Remod_Add` in plain words, and say what that pile-up at 1950 does to how you should read the left end of the curve (what is the 1950 group really made of?). (b) Compare the spline with the linear model for `Total_Bsmt_SF` at 0, 3,000, and 4,000 sq ft. Where do they differ most, and how many homes are out there — which one are you more worried about, and why? (c) A GAM gives each predictor its own curve but makes the effects *add up*. Give a concrete example from housing of an interaction that an additive model cannot express, and say what you'd add to capture it. (d) What do you give up in interpretability moving from the linear model to the GAM, and what do you keep?

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Pick the right tool
*Role lean: DS · Causal · ML*

For each situation, choose among: straight line, log transform, polynomial, spline / GAM, or tree-based model — and give a one- or two-sentence justification: (1) a regulator wants a defensible statement about how price changes per additional bathroom; (2) a pricing team wants the best possible predictions of sale price from 80 mixed features and doesn't care about interpretation; (3) you're doing an analysis for a decision-science team where the effect of building age on price clearly bends, and you need to show the shape to executives; (4) you plan to use the fitted model to *extrapolate* year-built effects to homes built in the future. For (4), say what's wrong with each of your instinctive choices.

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_